In [ ]:
# Uninstall the currently installed Transformers library
!pip uninstall -y transformers

# Install a specific version of Transformers (4.46.3)
# along with its required dependencies
!pip install -U transformers==4.46.3 \
                accelerate \
                sentencepiece \
                huggingface-hub==0.26.2

In [ ]:
# Load the load_dataset from datasets library
from datasets import load_dataset
# Load our rotten_tomatoes movie review dataset
data =load_dataset("rotten_tomatoes")
# display the data
data

In [ ]:
# Get the first and last examples from the training dataset
data["train"][[0, -1]]

In [ ]:
# Uninstall any existing versions of TensorFlow, Keras, and tf-keras
# This helps avoid version conflicts.
#TensorFlow is the engine that performs all the mathematical computations needed to train and run AI models.
!pip uninstall -y tf-keras keras tensorflow tensorflow-cpu

# Install TensorFlow version 2.20.0
# Install tf-keras, which provides compatibility with some Hugging Face models
!pip install tensorflow==2.20.0 tf-keras

In [ ]:
#Shows the tenser flow keras version
import tf_keras
print(tf_keras.__version__)

In [ ]:
# Import pipeline from transformers library
# A pipeline is used to take our input and process it using  a model and return the generated text
from transformers import pipeline
# Path to our HF model
# Roberta is a pretained for  twitter sentiment analysis

model_path = "cardiffnlp/twitter-roberta-base-sentiment-latest"
# Load model into pipeline
# Create a sentiment analysis pipeline using the pretrained model.
pipe = pipeline(
# loads the previously loaded model (Roberta)
model=model_path,
#Load the tokenizer attached to the model
tokenizer=model_path,
# Return the scores for all sentiment classes
# (e.g., Positive, Neutral, and Negative)
return_all_scores=
True
,
# connect device to CPU
device=-1
)

In [ ]:
# Import numpy for numerical calculations
import numpy as np
# Import tqdm library to display the progress bar
from tqdm import tqdm
# Import KeyDataset to efficiently pass only the required column
from transformers.pipelines.pt_utils import KeyDataset
#creates a list to store the predicted labels
y_pred = []
#Run sentiment analysis on the loaded dataset
#KeyDataset extract only text column for analysis
# tqdm display the progress bar
for output in tqdm(pipe(KeyDataset(data["test"], "text")), total=len(data["test"])):

# Get the probability socres dor negative and positve
   negative_score = output[0]["score"]
   positive_score = output[2]["score"]
# Compare the 2 scores and note the index of larger score

   assignment = np.argmax([negative_score, positive_score])
  # add the predicted label to the created list
   y_pred.append(assignment)

In [ ]:
# Import classification_report from sklearn metrics
#It is used to evaluate how well our model performed
from sklearn.metrics import classification_report
# creates a function
def evaluate_performance(y_true, y_pred):
# Create and print the classification report
  performance = classification_report(
  # loads the y actual and y predicted values which are previously stored
      y_true, y_pred,
  #Give meaning full names  to the datset
  # 0 - Negative
  #1 -Positive
      target_names=["Negative Review", "Positive Review"])
  # print the classificatin report
  print(performance)
  # call the function to evaluate the predictions
evaluate_performance(data["test"]["label"], y_pred)

In [ ]:
# Import sentencetransformer from sentence_transformers
# It is used to load the pretrained embedding models that convert text into embeddings
from sentence_transformers import SentenceTransformer
# Load model
# "all-mpnet-base-v2" is a popular model that generates high-quality sentence embeddings.
model = SentenceTransformer("sentence-transformers/all-mpnet-base-v2")
# Convert text to embeddings
train_embeddings = model.encode(data["train"]["text"],
# show_progress_bar=True displays a progress bar while encoding the data.
show_progress_bar=
True
)
# Convert all the test reviews into embeddings
test_embeddings = model.encode(data["test"]["text"],
show_progress_bar=
True
)

In [ ]:
# Shows the shape of train_ebeddings
# 8530 Number of training reviews (rows)
# 768- Number of values in each embedding vector (columns)
train_embeddings.shape


In [ ]:
# Import Logistice regression from sklearn,linear_model
# Logistice regression is used for classification task
from sklearn.linear_model import LogisticRegression

# Create a Logistic Regression model.
# random_state=42 ensures that the results are reproducible
clf = LogisticRegression(random_state=42)
# Train a logistic regression on our  actual train embeddings and labels (0-negative ,1-Positive )
clf.fit(train_embeddings, data["train"]["label"])
# Predict previously unseen instances
y_pred = clf.predict(test_embeddings)
# Evaluate the predictions by comparing them
evaluate_performance(data["test"]["label"], y_pred)

In [ ]:
# Create a text-to-text generation pipeline using the FLAN-T5 model.
# load the text2text model .Which tale input as text and give output in text
pipe = pipeline("text2text-generation",
 # Load the pretrained FLAN-T5 Small model from Hugging Face.
# FLAN-T5 is an instruction-tuned model that can perform tasks like summarization, translation, question answering, and classification.
model="google/flan-t5-small",
# connect to CPU
device=-1
)

In [ ]:
# Prepare our data
prompt = "Is the following sentence positive or negative? "
#map() is a function that performs the same operation on every example (row) in the dataset.
# add a new column called t5 after mapping like "This review is postive or negative  + movie review "
data = data.map(
lambda
 example: {"t5": prompt + example['text']})
# display the data
data

In [ ]:
# creates a empty list to store the predicted labels
y_pred = []
# KeyDataset extracts the "t5" column, which contains the prompt and the movie review.
# tqdm displays a progress bar while processing.

for output in tqdm(pipe(KeyDataset(data["test"], "t5")), total=len(data["test"])):
  #stores the generated text like positive or negaitve
    text = output[0]["generated_text"]
    #Convert the text prediction into numerical number
    y_pred.append(0
if
 text == "negative"
else
 1)

In [ ]:
# Compare the model's predicted labels with the actual (true) labels in the test dataset and evaluate the model's performance.
evaluate_performance(data["test"]["label"], y_pred)

In [ ]:
# Install or upgrade the Google GenAI
!pip install -U google-genai

In [ ]:
#import genAI from google
from google import genai

# Create a Gemini client using your API key
# Replace the API key with your own valid key.
client = genai.Client(api_key="YOUR KEY")


# Function to generate a response from Gemini
def chatgpt_generation(prompt, document, model="models/gemini-flash-latest"):
    """Generate an output based on a prompt and an input document."""

    # Create a conversation with a system message and a user message
    messages = [
        {
            "role": "system",
            "content": "You are a helpful assistant."
        },
        {
            "role": "user",
            # Replace [DOCUMENT] with the actual document
            "content": prompt.replace("[DOCUMENT]", document)
        }
    ]

    # Gemini accepts text, so create one text string
      # Create an empty string to store the conversation.
    conversation = ""

    # Add each message to the conversation
    # like role :user ,content :"It is helpful "
    # It will display like user :It is helpful
    for message in messages:
        conversation += f"{message['role']}: {message['content']}\n"

    # Send the conversation to the Gemini model
    response = client.models.generate_content(
        model=model,
        contents=conversation
    )

    # Return only the generated text
    return response.text.strip()


# Prompt template
prompt = """
Predict whether the following document is a positive or negative movie review:

[DOCUMENT]

If it is positive return 1 and if it is negative return 0.
Do not give any other answers.
"""

# Input document
document = "unpretentious, charming, quirky, original"

# Call the function and print the prediction using document input
print(chatgpt_generation(prompt, document))

In [ ]:
#  YEXT CLASSIFICATIONIS DONE IN ABOVE THIS IS WITHOUT USING THE OPENAI like gpt ot gemini

In [ ]:
# creates a function to predict and load the model
def chatgpt_generation(prompt, document, model="google/flan-t5-small"):
    """Generate an output based on a prompt and an input document."""
# Create a conversation with the system and user
    messages = [
        {
            "role": "system",
            "content": "You are a helpful assistant."
        },
        {
            "role": "user",
            "content": prompt.replace("[DOCUMENT]", document)
        }
    ]
# Extract only the user's prompt from the messages list.
# messages[0] is the system message.
# messages[1] is the user message.
    generation = pipe(
        messages[1]["content"],
        # generate only 50 new tokens
        max_new_tokens=50,
        # Disable random sampling
        do_sample=False
    )
# returns the generated text
    return generation[0]["generated_text"]

In [ ]:
prompt = """Predict whether the following document is a positive
or negative movie review:

[DOCUMENT]

If it is positive return 1 and if it is negative return 0.
Do not give any other answers.
"""

document = "unpretentious, charming, quirky, original"
# call the function and print the prediction
print(chatgpt_generation(prompt, document))

In [ ]:
# You can skip this if you want to save your (free) credits
predictions = [
    # Call the chatgpt_generation() function for each movie review (doc).
    # 1 - Positive review
    # 0 - Negative review
chatgpt_generation(prompt, doc)
# tqdm display the progress bar
    # Loop through every movie review in the "text" column
    # of the test dataset
for doc in tqdm(data["test"]["text"])]

In [ ]:
# Extract predictions
y_pred = [
# The model may return:
# "positive"  -> Convert to 1
# "negative"  -> Convert to 0
# "1" or "0" -> Convert the string to an integer
    1 if pred.lower() == "positive"
    else 0 if pred.lower() == "negative"
    # Convert the strings into integers like "0"to 0
    else int(pred)
    # stores in the predictions list
    for pred in predictions
]
# Compare the predicted labels with the actual labels
# in the test dataset and display the model's performance.
# Evaluate performance
evaluate_performance(data["test"]["label"], y_pred)
